In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision 
from torchvision.datasets import CIFAR10

In [5]:
# Datasets And Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image => scale(0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


trainset = CIFAR10(root = "./data",train=True,download=True,transform=transform) 
testset = CIFAR10(root = "./data",train=False,download=True,transform=transform) 

In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader =  DataLoader(testset,batch_size=64)

### Build The CNN

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size=2, stride value=2
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening step
        x = self.fc_layers(x)

        return x #flattening step

In [9]:
model = CNN()

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training CNN

In [12]:
epochs = 10

for epoch in range (epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        
        output = model.forward(images) #FP
        loss = criterion(output,labels) #Loss Computation
        loss.backward() #BP
        optimizer.step() #update parameters 

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")        

epoch = 1/10 & loss = 1.3688386292256358
epoch = 2/10 & loss = 0.9306924263077319
epoch = 3/10 & loss = 0.7433224454941347
epoch = 4/10 & loss = 0.6152005744407244
epoch = 5/10 & loss = 0.5067496218187425
epoch = 6/10 & loss = 0.4104904384373704
epoch = 7/10 & loss = 0.3243383255684772
epoch = 8/10 & loss = 0.25805160665736937
epoch = 9/10 & loss = 0.19218197103847018
epoch = 10/10 & loss = 0.15137789919591316


In [17]:
# Evaluvate 

correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images,labels in testloader:
        output = model.forward(images)
        _, predicted = torch.max(output,1)

        correct_labels += (predicted == labels ).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels * 100}")
    

accuracy = 75.32
